# Extracción, limpieza y augmentation del dataset AQS-EPA

**Curso:** Programación Concurrente y Distribuida — CC65 — UPC
**Trabajo Parcial — 2026-01**

---

Este notebook ejecuta la **Fase 1** del proyecto: preparar el dataset que después
consumirá la solución concurrente y distribuida en Go.

**Pipeline:**
1. Descarga de 22 archivos `annual_conc_by_monitor_YYYY.zip` (años 2004–2025) desde el AQS-EPA
2. Carga y consolidación en un DataFrame
3. Filtrado por los 4 parámetros de interés (todo Estados Unidos — *Path B*)
4. Limpieza (3 reglas)
5. **Checkpoint**: export del DataFrame limpio en CSV
6. Data augmentation (bootstrap + ruido gaussiano) hasta 3,000,000 de filas
7. Validación pre/post augmentation
8. Export final en CSV

**Fuente:** https://aqs.epa.gov/aqsweb/airdata/download_files.html

## 1. Setup

In [ ]:
# Instalación de paquetes (la mayoría ya están en Colab por defecto).
!pip install -q tqdm requests pandas numpy

In [ ]:
import os
import io
import zipfile
import time
import shutil
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

print(f'pandas {pd.__version__} | numpy {np.__version__}')

## 2. Configuración

Variables centralizadas. Si necesitas cambiar parámetros, años, o rutas, modifica solo esta celda.

In [ ]:
# --- Parámetros AQS (códigos oficiales) ---
POLLUTANTS = {
    '88101': 'PM2.5',
    '42602': 'NO2',
    '44201': 'O3',
    '42101': 'CO',
}
TARGET_PARAM_CODES = list(POLLUTANTS.keys())

# --- Estándar primario por parámetro (el más reciente) ---
# Sirve para deduplicar: cada monitor-año aparece N veces, una por Pollutant Standard.
# Nos quedamos con un único estándar de referencia por parámetro.
PRIMARY_STANDARDS = {
    '88101': 'PM25 Annual 2024',     # PM2.5 anual
    '42602': 'NO2 Annual 1971',      # NO2 anual
    '44201': 'Ozone 8-hour 2015',    # O3 8 horas (estándar más reciente)
    '42101': 'CO 8-hour 1971',       # CO 8 horas
}

# --- Event Types a conservar (descartamos las versiones 'Excluded') ---
KEEP_EVENT_TYPES = {'No Events', 'Events Included'}

# --- Rango temporal ---
YEAR_START = 2004
YEAR_END = 2025  # inclusive  -> 22 años
YEARS = list(range(YEAR_START, YEAR_END + 1))

# --- URL pattern oficial EPA ---
AQS_URL_TEMPLATE = 'https://aqs.epa.gov/aqsweb/airdata/annual_conc_by_monitor_{year}.zip'

# --- Target tamaño dataset final (post augmentation) ---
TARGET_ROWS = 1_000_000

# --- Augmentation: σ relativa para ruido gaussiano por columna ---
# σ_efectivo = NOISE_SIGMA_RATIO * std(columna)
NOISE_SIGMA_RATIO = 0.05  # 5% de la desviación estándar real

# --- Random seed para reproducibilidad ---
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f'Parámetros: {list(POLLUTANTS.values())}')
print(f'Años: {YEAR_START}-{YEAR_END} ({len(YEARS)} archivos)')
print(f'Target final: {TARGET_ROWS:,} filas')

## 3. Rutas de salida

Los CSVs se generan en la carpeta `data/` relativa al notebook.
Puedes cambiar `BASE_DIR` si quieres guardarlos en otro lugar.

In [ ]:
# Ejecucion local — no se necesita montar Google Drive.
# Si en el futuro lo ejecutas en Colab, descomenta las 2 lineas de abajo:
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
import os

# Ruta base: carpeta 'data/' al lado del notebook (se crea si no existe)
BASE_DIR = Path(os.path.dirname(os.path.abspath('__file__'))) / 'data'

RAW_DIR   = BASE_DIR / 'raw'               # zips descargados
CLEAN_CSV = BASE_DIR / 'aqs_clean.csv'     # checkpoint post-limpieza
FINAL_CSV = BASE_DIR / 'aqs_final_3M.csv'  # dataset final con augmentation

for d in (BASE_DIR, RAW_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f'Directorio de trabajo: {BASE_DIR}')


## 4. Descarga de archivos Annual Summary

Cada archivo es un zip que contiene un CSV con el resumen anual de TODOS los monitores
y TODOS los parámetros para ese año a nivel nacional.

**Idempotente**: si ya existe el zip en disco, se omite la descarga.

In [ ]:
def download_file(url: str, dest: Path, max_retries: int = 3, timeout: int = 120) -> bool:
    """Descarga con reintentos y caché local."""
    if dest.exists() and dest.stat().st_size > 1024:
        return True  # ya descargado

    tmp = dest.with_suffix(dest.suffix + '.part')
    for attempt in range(1, max_retries + 1):
        try:
            with requests.get(url, stream=True, timeout=timeout) as r:
                r.raise_for_status()
                with open(tmp, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=1 << 16):
                        f.write(chunk)
            tmp.rename(dest)
            return True
        except requests.RequestException as e:
            print(f'  Intento {attempt}/{max_retries} falló: {e}')
            if tmp.exists():
                tmp.unlink()
            if attempt < max_retries:
                time.sleep(5 * attempt)
    return False

In [ ]:
# Descargar los 22 archivos
downloaded = []
missing = []
for year in tqdm(YEARS, desc='Descargando'):
    url = AQS_URL_TEMPLATE.format(year=year)
    dest = RAW_DIR / f'annual_conc_by_monitor_{year}.zip'
    ok = download_file(url, dest)
    (downloaded if ok else missing).append(year)

print(f'Descargados/cacheados: {len(downloaded)} archivos')
if missing:
    print(f'⚠ NO se pudieron descargar: {missing}')

total_size_mb = sum(p.stat().st_size for p in RAW_DIR.glob('*.zip')) / 1e6
print(f'Tamaño total en disco: {total_size_mb:.1f} MB')

## 5. Carga, consolidación y filtrado

Estrategia para no agotar RAM en Colab: leer cada zip, filtrar inmediatamente a los
4 parámetros de interés (descarta ~95% de las filas), y solo después concatenar.

Esto es **Path B**: todos los monitores de Estados Unidos, no solo las 4 ciudades análogas.
Las 4 ciudades se usarán más adelante como conjunto de validación/análogo a Lima.

In [ ]:
# Columnas que SÍ se cargan (descartamos administrativas y redundantes para ahorrar RAM)
USECOLS = [
    'State Code', 'County Code', 'Site Num',
    'Parameter Code', 'POC',
    'Latitude', 'Longitude',
    'Parameter Name', 'Sample Duration', 'Pollutant Standard',
    'Year', 'Units of Measure', 'Event Type',
    'Observation Count', 'Observation Percent', 'Completeness Indicator',
    'Valid Day Count', 'Required Day Count',
    'Exceptional Data Count', 'Null Data Count',
    'Primary Exceedance Count', 'Secondary Exceedance Count',
    'Num Obs Below MDL',
    'Arithmetic Mean', 'Arithmetic Standard Dev',
    '1st Max Value', '2nd Max Value', '3rd Max Value', '4th Max Value',
    '99th Percentile', '98th Percentile', '95th Percentile',
    '90th Percentile', '75th Percentile', '50th Percentile', '10th Percentile',
    'State Name', 'County Name', 'City Name', 'CBSA Name',
]

# Tipos para evitar warnings y pérdida de leading zeros en los códigos
DTYPES = {
    'State Code': str, 'County Code': str, 'Site Num': str,
    'Parameter Code': str,
}

In [ ]:
def load_and_filter_year(year: int) -> pd.DataFrame:
    """Lee el zip de un año, filtra a los 4 parámetros, devuelve DataFrame."""
    zip_path = RAW_DIR / f'annual_conc_by_monitor_{year}.zip'
    if not zip_path.exists():
        return pd.DataFrame()

    with zipfile.ZipFile(zip_path) as zf:
        csv_name = next(n for n in zf.namelist() if n.lower().endswith('.csv'))
        with zf.open(csv_name) as fh:
            df = pd.read_csv(fh, usecols=USECOLS, dtype=DTYPES, low_memory=False)

    # Filtro inmediato a los 4 parámetros (reduce tamaño ~20x)
    df = df[df['Parameter Code'].isin(TARGET_PARAM_CODES)].copy()
    return df

# Cargar todos los años
parts = []
for year in tqdm(YEARS, desc='Cargando + filtrando'):
    parts.append(load_and_filter_year(year))

df_raw = pd.concat(parts, ignore_index=True)
del parts
print(f'Filas tras filtro por parámetros: {len(df_raw):,}')
print(f'Memoria del DataFrame: {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB')
df_raw.head(3)

## 6. Limpieza

Tres reglas:

1. **Completeness Indicator = `Y`** — solo monitor-años con datos completos según criterios EPA.
2. **Event Type ∈ {No Events, Events Included}** — descarta variantes 'Excluded' que duplican filas.
3. **Pollutant Standard primario** — cada monitor-año aparece múltiples veces (una por estándar);
   conservamos solo el estándar más reciente por parámetro.

In [ ]:
df = df_raw.copy()
print(f'Inicial: {len(df):,} filas')

# Regla 1: Completeness = Y
df = df[df['Completeness Indicator'] == 'Y']
print(f'Tras Completeness=Y: {len(df):,} filas')

# Regla 2: Event Type
df = df[df['Event Type'].isin(KEEP_EVENT_TYPES)]
print(f'Tras Event Type: {len(df):,} filas')

# Regla 3: Pollutant Standard primario por parámetro
primary_mask = df.apply(
    lambda r: r['Pollutant Standard'] == PRIMARY_STANDARDS.get(r['Parameter Code']),
    axis=1,
)
df = df[primary_mask]
print(f'Tras Pollutant Standard primario: {len(df):,} filas')

# Bonus: si todavía quedan duplicados (mismo monitor-año-parametro), promediar
key = ['State Code', 'County Code', 'Site Num', 'POC', 'Parameter Code', 'Year']
before = len(df)
df = df.groupby(key, as_index=False).agg({
    **{c: 'first' for c in ['Latitude', 'Longitude', 'Parameter Name',
                            'Sample Duration', 'Pollutant Standard',
                            'Units of Measure', 'Event Type',
                            'State Name', 'County Name', 'City Name', 'CBSA Name']},
    **{c: 'mean' for c in ['Observation Count', 'Observation Percent',
                            'Valid Day Count', 'Required Day Count',
                            'Exceptional Data Count', 'Null Data Count',
                            'Primary Exceedance Count', 'Secondary Exceedance Count',
                            'Num Obs Below MDL',
                            'Arithmetic Mean', 'Arithmetic Standard Dev',
                            '1st Max Value', '2nd Max Value', '3rd Max Value', '4th Max Value',
                            '99th Percentile', '98th Percentile', '95th Percentile',
                            '90th Percentile', '75th Percentile', '50th Percentile', '10th Percentile']},
})
print(f'Tras dedup por monitor-año-parámetro: {len(df):,} filas (antes: {before:,})')

# Agregar columna legible con el nombre del contaminante
df['pollutant'] = df['Parameter Code'].map(POLLUTANTS)

df.head(3)

In [ ]:
# Resumen de cobertura por parámetro × año
cobertura = df.groupby(['pollutant', 'Year']).size().unstack(fill_value=0)
print('Filas por contaminante × año:')
cobertura

## 7. ⛳ Checkpoint — Export del dataset limpio

Aquí se guarda el DataFrame **limpio pero sin augmentation**. Esto te permite re-ejecutar
la augmentation con diferentes parámetros sin volver a hacer el download + limpieza.

In [ ]:
df.to_csv(CLEAN_CSV, index=False)
size_mb = CLEAN_CSV.stat().st_size / 1e6
print(f'✓ Checkpoint guardado: {CLEAN_CSV}')
print(f'  {len(df):,} filas × {len(df.columns)} columnas | {size_mb:.1f} MB')

## 8. Data Augmentation

Objetivo: llegar a **3,000,000 filas** combinando dos técnicas:

- **Bootstrap resampling**: muestreo con reemplazo de las filas reales hasta alcanzar el target.
- **Ruido gaussiano** sobre columnas numéricas, con σ proporcional a la std real de cada columna
  (`σ_ruido = NOISE_SIGMA_RATIO × std_columna`).

Se preservan **sin perturbar** las columnas identificadoras (State, County, Site, POC,
Parameter Code, Year) y las categóricas. Solo se perturban las columnas numéricas
(percentiles, means, max values, conteos).

Cada fila lleva una bandera `is_synthetic` para trazabilidad.

In [ ]:
# Columnas numéricas a perturbar
NUMERIC_COLS = [
    'Observation Count', 'Observation Percent',
    'Valid Day Count', 'Required Day Count',
    'Exceptional Data Count', 'Null Data Count',
    'Primary Exceedance Count', 'Secondary Exceedance Count',
    'Num Obs Below MDL',
    'Arithmetic Mean', 'Arithmetic Standard Dev',
    '1st Max Value', '2nd Max Value', '3rd Max Value', '4th Max Value',
    '99th Percentile', '98th Percentile', '95th Percentile',
    '90th Percentile', '75th Percentile', '50th Percentile', '10th Percentile',
]

# Columnas que NO se tocan (preserva grouping y categóricas)
PRESERVE_COLS = [c for c in df.columns if c not in NUMERIC_COLS]

# Std real por columna (para calibrar el ruido)
col_stds = df[NUMERIC_COLS].std(numeric_only=True).fillna(0)
noise_sigmas = (col_stds * NOISE_SIGMA_RATIO).values  # vector de σ por columna
print('σ del ruido por columna:')
print(pd.Series(noise_sigmas, index=NUMERIC_COLS).round(4))

In [ ]:
def augment_to_target(df_base: pd.DataFrame, target_rows: int) -> pd.DataFrame:
    """Bootstrap + ruido gaussiano hasta alcanzar target_rows."""
    n_base = len(df_base)
    n_synthetic = max(0, target_rows - n_base)
    print(f'Filas reales: {n_base:,}  →  sintéticas a generar: {n_synthetic:,}')

    if n_synthetic == 0:
        out = df_base.copy()
        out['is_synthetic'] = False
        return out

    # Bootstrap: muestrear con reemplazo
    idx = np.random.randint(0, n_base, size=n_synthetic)
    synth = df_base.iloc[idx].reset_index(drop=True).copy()

    # Ruido gaussiano vectorizado sobre las columnas numéricas
    noise = np.random.normal(
        loc=0.0,
        scale=noise_sigmas,
        size=(n_synthetic, len(NUMERIC_COLS)),
    )
    synth[NUMERIC_COLS] = synth[NUMERIC_COLS].values + noise

    # Constraints físicos: ningún valor numérico puede ser negativo (concentraciones)
    for c in NUMERIC_COLS:
        synth[c] = synth[c].clip(lower=0)

    # Banderas
    df_base = df_base.copy()
    df_base['is_synthetic'] = False
    synth['is_synthetic'] = True

    return pd.concat([df_base, synth], ignore_index=True)

df_final = augment_to_target(df, TARGET_ROWS)
print(f'Tamaño dataset final: {len(df_final):,} filas')
print(f'  Reales:     {(~df_final["is_synthetic"]).sum():,}')
print(f'  Sintéticas: {df_final["is_synthetic"].sum():,}')

## 9. Validación pre/post augmentation

Comparamos estadísticas básicas para confirmar que la augmentation no distorsiona
la distribución original más allá del ruido esperado.

In [ ]:
real = df_final[~df_final['is_synthetic']]
synth = df_final[df_final['is_synthetic']]

summary_cols = ['Arithmetic Mean', '99th Percentile', '50th Percentile', '1st Max Value']
comp = pd.concat({
    'Real':      real[summary_cols].describe().loc[['mean', 'std', '50%']],
    'Sintética': synth[summary_cols].describe().loc[['mean', 'std', '50%']],
}, axis=1).round(4)
print('Comparación de estadísticas (real vs sintética):')
comp

In [ ]:
# Cobertura por contaminante en el dataset final
print('Distribución por contaminante en dataset final:')
df_final.groupby('pollutant').agg(
    n_total=('Year', 'size'),
    n_real=('is_synthetic', lambda s: (~s).sum()),
    n_synth=('is_synthetic', 'sum'),
)

## 10. Export final

Dataset final en CSV. Este es el archivo que consumirá la solución concurrente y distribuida en Go.

In [ ]:
df_final.to_csv(FINAL_CSV, index=False)
size_mb = FINAL_CSV.stat().st_size / 1e6
print(f'✓ Dataset final guardado: {FINAL_CSV}')
print(f'  {len(df_final):,} filas × {len(df_final.columns)} columnas | {size_mb:.1f} MB')